# Segmentación interactiva con MedSAM2

En la [clase 4](../clase4.qmd) contrastamos dos paradigmas de segmentación: **automática** (el modelo produce una máscara solo a partir de la imagen, como nnU-Net) e **interactiva** (el experto aporta un *prompt* — un punto, una caja — y el modelo lo usa para delinear la estructura).

Este notebook explora el segundo paradigma con [MedSAM2](https://github.com/bowang-lab/MedSAM2), la adaptación al dominio médico de SAM2. La idea central de MedSAM2 es que un volumen de TC puede tratarse como una **secuencia de video**: cada corte axial es un "frame", y el mismo mecanismo que SAM2 usa para seguir un objeto a lo largo de un video (dar un prompt en un frame y propagarlo al resto) sirve para segmentar una estructura en las tres dimensiones.

Vamos a:

1. Cargar un caso de TC (reutilizando el conjunto **Task09_Spleen** ya usado en el [Notebook1](Notebook1.ipynb) y el [Notebook6](Notebook6.ipynb)).
2. Construir una interfaz interactiva (Gradio) para dar distintos tipos de *prompt* — punto positivo, punto negativo, caja — sobre un corte.
3. Propagar ese prompt al resto del volumen, tal como MedSAM2 propaga máscaras en un video.
4. Comparar cuantitativamente el resultado *zero-shot* contra el *ground truth* conocido del caso.

::: {.callout-important}
## Este notebook requiere GPU

En Colab: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`. Sin GPU, MedSAM2 puede ejecutarse pero cada segmentación tardará mucho más.
:::

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ignacio-scarinci/ia-fisica-medica/blob/main/unidad2/notebooks/Notebook7.ipynb)

## Preparación del entorno

MedSAM2 no se distribuye como paquete independiente en PyPI: el paquete `sam2` vive dentro del propio repositorio. Por eso, en lugar de un simple `pip install`, clonamos el repositorio e lo instalamos en modo editable. También descargamos únicamente el checkpoint `MedSAM2_latest.pt` (en vez de correr `download.sh`, que baja varios checkpoints de varios GB en total).

In [ ]:
import os

REPO_DIR = "MedSAM2"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/bowang-lab/MedSAM2.git {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .
!pip install -q "gradio>=4.44.0" opencv-python-headless
!python -c "import monai" || pip install -q "monai-weekly[nibabel]"
!python -c "import nibabel" || pip install -q nibabel

os.makedirs("checkpoints", exist_ok=True)
CKPT_PATH = "checkpoints/MedSAM2_latest.pt"
CKPT_URL = "https://huggingface.co/wanglab/MedSAM2/resolve/main/MedSAM2_latest.pt"
if not os.path.exists(CKPT_PATH):
    !wget -q -O {CKPT_PATH} {CKPT_URL}

# config de la variante "tiny" del encoder Hiera, la más liviana disponible
MODEL_CONFIG = "configs/sam2.1_hiera_t512.yaml"

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("Aviso: no se detectó GPU. MedSAM2 puede correr en CPU, pero cada segmentación será mucho más lenta.")
print(f"Dispositivo: {device}")

## Datos: un caso de Task09_Spleen

Reutilizamos el mismo conjunto de datos y el mismo mecanismo de descarga que en el `Notebook1` y el `Notebook6`. La ventaja de reusar este dataset es que ya conocemos su *ground truth*, así que podemos comparar el resultado *zero-shot* de MedSAM2 contra una referencia conocida.

In [ ]:
import glob
import tempfile

from monai.apps import download_and_extract

directory = os.environ.get("MONAI_DATA_DIRECTORY")
if directory is not None:
    os.makedirs(directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if directory is None else directory

resource = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar"
md5 = "410d4a301da4e5b2f6f86ec3ddba524e"
compressed_file = os.path.join(root_dir, "Task09_Spleen.tar")
data_dir = os.path.join(root_dir, "Task09_Spleen")
if not os.path.exists(data_dir):
    download_and_extract(resource, compressed_file, root_dir, md5)

train_images = sorted(glob.glob(os.path.join(data_dir, "imagesTr", "*.nii.gz")))
train_labels = sorted(glob.glob(os.path.join(data_dir, "labelsTr", "*.nii.gz")))

# el mismo caso reservado para validación en el Notebook1/Notebook6
case_image, case_label = train_images[-1], train_labels[-1]
print(f"Caso elegido: {os.path.basename(case_image)}")

## De volumen 3D a "video" de cortes

MedSAM2 espera una carpeta de imágenes JPEG numeradas, exactamente como si fueran los *frames* de un video. Aplicamos una ventana de tejido blando para llevar la TC a un rango de intensidades visualmente interpretable y exportamos únicamente los cortes donde hay bazo (identificados a partir de la máscara de referencia, solo para acotar el rango — el modelo no ve esta información).

In [ ]:
import nibabel as nib
import numpy as np
from PIL import Image

img_vol = nib.load(case_image).get_fdata()  # (H, W, D)
lbl_vol = nib.load(case_label).get_fdata()

# ventana de tejido blando (nivel 40 HU, ancho 400 HU)
WL, WW = 40, 400
lo, hi = WL - WW / 2, WL + WW / 2
img_uint8 = np.clip((img_vol - lo) / (hi - lo), 0, 1)
img_uint8 = (img_uint8 * 255).astype(np.uint8)

spleen_slices = np.where(lbl_vol.sum(axis=(0, 1)) > 0)[0]
slice_start, slice_end = int(spleen_slices.min()), int(spleen_slices.max())
print(f"Bazo presente entre los cortes {slice_start} y {slice_end} (de {img_vol.shape[-1]} en total)")

frames_dir = os.path.join(root_dir, "medsam2_frames")
os.makedirs(frames_dir, exist_ok=True)
for local_idx, z in enumerate(range(slice_start, slice_end + 1)):
    slice_rgb = np.stack([img_uint8[:, :, z]] * 3, axis=-1)
    slice_rgb = np.rot90(slice_rgb)  # orientación radiológica habitual al graficar
    Image.fromarray(slice_rgb).save(os.path.join(frames_dir, f"{local_idx:05d}.jpg"))

n_frames = slice_end - slice_start + 1
print(f"{n_frames} cortes exportados en {frames_dir}")

## Construcción de los predictores

MedSAM2 (a través de SAM2) expone dos interfaces distintas para el mismo modelo:

- `SAM2ImagePredictor`: segmenta un único corte a partir de un prompt, sin memoria entre cortes.
- `build_sam2_video_predictor`: mantiene un estado de inferencia (`inference_state`) y **propaga** una máscara inicial a lo largo de todos los "frames" (en nuestro caso, cortes).

Usamos la primera para segmentar rápidamente el corte de referencia y la segunda para propagar esa segmentación al resto del volumen.

In [ ]:
from sam2.build_sam import build_sam2, build_sam2_video_predictor
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_model = build_sam2(MODEL_CONFIG, CKPT_PATH, device=device)
image_predictor = SAM2ImagePredictor(sam2_model)

video_predictor = build_sam2_video_predictor(MODEL_CONFIG, CKPT_PATH, device=device)
inference_state = video_predictor.init_state(video_path=frames_dir)

# corte de referencia: el del medio del rango exportado
ref_frame_idx = n_frames // 2
frame_paths = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
ref_image = np.array(Image.open(frame_paths[ref_frame_idx]))
print(f"Predictores listos. Corte de referencia: frame {ref_frame_idx} de {n_frames}.")

## Interfaz interactiva: múltiples órganos con distintos colores

La siguiente celda arma una interfaz Gradio embebida, similar en espíritu a la app del repositorio (`app.py`), pero reducida a lo esencial para esta clase. Cada "objeto" (por ejemplo, un órgano distinto) tiene su propio color y su propio conjunto de prompts, tal como `add_new_points_or_box` de SAM2 admite un `obj_id` por estructura:

1. Elegí el **objeto activo** (define el color con el que se dibuja y se superpone su máscara).
2. Elegí el tipo de prompt (punto positivo, punto negativo o caja) con el selector.
3. Hacé click sobre la imagen de la izquierda para agregarlo al objeto activo. **Para una caja: dos clicks, no arrastres** (primero una esquina, después la esquina opuesta) — `gr.Image` no soporta arrastrar para dibujar un rectángulo, y arrastrar termina moviendo la imagen en el navegador.
4. Repetí 1-3 para cada estructura que quieras segmentar en el mismo corte.
5. **Segmentar este corte** corre `SAM2ImagePredictor` una vez por cada objeto con prompts cargados, y superpone cada máscara con su color.
6. **Propagar por el volumen** inicializa el `video_predictor` con los prompts de todos los objetos a la vez en el corte de referencia y propaga todas las máscaras juntas al resto de los cortes.

::: {.callout-note}
## Nota

Podés combinar varios puntos (positivos y negativos) antes de segmentar, pero solo se admite **una** caja por objeto: un segundo click en modo "Caja" reemplaza la anterior. "Reiniciar prompts" solo borra los prompts del objeto activo, no los del resto.
:::

::: {.callout-warning}
## Si las imágenes aparecen "rotas"

El notebook lanza la interfaz con `share=True`: el servidor de Gradio corre localmente, pero el iframe embebido en la celda de salida (tanto en Colab como en VS Code/Jupyter) no siempre puede llegar a `127.0.0.1`, así que se necesita un túnel público (`*.gradio.live`). Si aun así ves íconos de imagen rotos, esperá unos segundos a que el túnel termine de establecerse, o volvé a correr la celda.
:::

In [ ]:
import cv2
import gradio as gr

# paleta fija: un color distinto por objeto (rojo, verde, azul, naranja)
OBJECT_COLORS = {1: (230, 25, 75), 2: (60, 180, 75), 3: (0, 130, 200), 4: (245, 130, 48)}
OBJECT_LABELS = {obj_id: f"Objeto {obj_id}" for obj_id in OBJECT_COLORS}

# evita que el navegador arrastre la imagen como si fuera un archivo al hacer click y mover
# (interfiere con el modo "Caja", que se completa con dos clicks, no arrastrando)
DISABLE_IMG_DRAG_CSS = """
#ref_image img, #result_image img {
    -webkit-user-drag: none;
    user-drag: none;
    user-select: none;
}
"""


def _empty_prompt():
    return {"points": [], "labels": [], "box": None, "box_start": None}


prompt_state = {obj_id: _empty_prompt() for obj_id in OBJECT_COLORS}
propagated_masks = {obj_id: {} for obj_id in OBJECT_COLORS}  # obj_id -> {frame_idx: mask}


def _obj_id_from_label(label):
    return int(label.split()[-1])


def active_object_ids():
    """Objetos con al menos un prompt cargado."""
    return [obj_id for obj_id, state in prompt_state.items() if state["points"] or state["box"] is not None]


def overlay_masks(image, masks_by_obj, alpha=0.5):
    overlay = image.copy()
    for obj_id, mask in masks_by_obj.items():
        region = mask.astype(bool)
        color = np.array(OBJECT_COLORS[obj_id])
        overlay[region] = ((1 - alpha) * overlay[region] + alpha * color).astype(np.uint8)
    return overlay


def draw_prompts(image):
    vis = image.copy()
    for obj_id, state in prompt_state.items():
        color = OBJECT_COLORS[obj_id]
        for (x, y), lab in zip(state["points"], state["labels"]):
            cv2.circle(vis, (int(x), int(y)), 4, color, -1 if lab == 1 else 2)
        if state["box"] is not None:
            (x0, y0), (x1, y1) = state["box"]
            cv2.rectangle(vis, (int(x0), int(y0)), (int(x1), int(y1)), color, 2)
        elif state["box_start"] is not None:
            # marca la primera esquina mientras se espera el segundo click
            cv2.drawMarker(vis, (int(state["box_start"][0]), int(state["box_start"][1])), color, cv2.MARKER_CROSS, 12, 2)
    return vis


def on_select(active_object, mode, evt: gr.SelectData):
    state = prompt_state[_obj_id_from_label(active_object)]
    x, y = evt.index
    if mode == "Caja":
        if state["box_start"] is None:
            state["box_start"] = (x, y)
            state["box"] = None
        else:
            state["box"] = (state["box_start"], (x, y))
            state["box_start"] = None
    else:
        label = 1 if mode == "Punto positivo" else 0
        state["points"].append((x, y))
        state["labels"].append(label)
    return draw_prompts(ref_image)


def reset_prompts(active_object):
    obj_id = _obj_id_from_label(active_object)
    prompt_state[obj_id] = _empty_prompt()
    propagated_masks[obj_id] = {}
    return draw_prompts(ref_image), ref_image, f"Prompts de {active_object} reiniciados."


def build_prompt_arrays(obj_id):
    state = prompt_state[obj_id]
    points = np.array(state["points"], dtype=np.float32) if state["points"] else None
    labels = np.array(state["labels"], dtype=np.int32) if state["labels"] else None
    box = None
    if state["box"] is not None:
        (x0, y0), (x1, y1) = state["box"]
        box = np.array([min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)], dtype=np.float32)
    return points, labels, box


def segment_slice():
    obj_ids = active_object_ids()
    if not obj_ids:
        return ref_image, "Agregá al menos un prompt en algún objeto antes de segmentar."
    masks_by_obj = {}
    for obj_id in obj_ids:
        points, labels, box = build_prompt_arrays(obj_id)
        image_predictor.set_image(ref_image)
        masks, scores, _ = image_predictor.predict(
            point_coords=points, point_labels=labels, box=box, multimask_output=False
        )
        masks_by_obj[obj_id] = masks[0]
    status = ", ".join(OBJECT_LABELS[obj_id] for obj_id in obj_ids)
    return overlay_masks(ref_image, masks_by_obj), f"Segmentados: {status}"


def show_propagated_slice(idx):
    idx = int(idx)
    frame = np.array(Image.open(frame_paths[idx]))
    masks_by_obj = {obj_id: masks[idx] for obj_id, masks in propagated_masks.items() if idx in masks}
    return frame if not masks_by_obj else overlay_masks(frame, masks_by_obj)


def propagate_volume():
    obj_ids = active_object_ids()
    if not obj_ids:
        return ref_image, "Agregá al menos un prompt en algún objeto antes de propagar."

    video_predictor.reset_state(inference_state)
    for obj_id in obj_ids:
        points, labels, box = build_prompt_arrays(obj_id)
        video_predictor.add_new_points_or_box(
            inference_state, frame_idx=ref_frame_idx, obj_id=obj_id, points=points, labels=labels, box=box
        )

    for obj_id in OBJECT_COLORS:
        propagated_masks[obj_id] = {}

    def _accumulate(frame_idx, ids, masks):
        for obj_id, mask in zip(ids, masks):
            propagated_masks[obj_id][frame_idx] = (mask[0] > 0).cpu().numpy()

    for frame_idx, ids, masks in video_predictor.propagate_in_video(inference_state):
        _accumulate(frame_idx, ids, masks)
    for frame_idx, ids, masks in video_predictor.propagate_in_video(inference_state, reverse=True):
        _accumulate(frame_idx, ids, masks)

    n_masks = sum(len(m) for m in propagated_masks.values())
    return show_propagated_slice(ref_frame_idx), f"Propagación completa para {len(obj_ids)} objeto(s), {n_masks} máscaras generadas."


with gr.Blocks(css=DISABLE_IMG_DRAG_CSS) as demo:
    gr.Markdown("### Hacé click sobre la imagen de la izquierda para agregar prompts")
    legend = ", ".join(f"{name} = RGB{OBJECT_COLORS[obj_id]}" for obj_id, name in OBJECT_LABELS.items())
    gr.Markdown(f"**Colores:** {legend}")
    gr.Markdown(
        '**Modo "Caja":** no arrastres — hacé un click en una esquina y otro click en la esquina opuesta.'
    )
    with gr.Row():
        active_object = gr.Radio(list(OBJECT_LABELS.values()), value=OBJECT_LABELS[1], label="Objeto activo")
        mode = gr.Radio(["Punto positivo", "Punto negativo", "Caja"], value="Punto positivo", label="Tipo de prompt")
    with gr.Row():
        image_display = gr.Image(
            value=ref_image, label=f"Corte de referencia (frame {ref_frame_idx})", height=400, elem_id="ref_image"
        )
        result_display = gr.Image(value=ref_image, label="Resultado", height=400, elem_id="result_image")
    status = gr.Textbox(label="Estado", interactive=False)
    with gr.Row():
        btn_reset = gr.Button("Reiniciar prompts del objeto activo")
        btn_segment = gr.Button("Segmentar este corte")
        btn_propagate = gr.Button("Propagar por el volumen")
    slice_slider = gr.Slider(0, n_frames - 1, value=ref_frame_idx, step=1, label="Corte del volumen propagado")

    image_display.select(on_select, inputs=[active_object, mode], outputs=image_display)
    btn_reset.click(reset_prompts, inputs=active_object, outputs=[image_display, result_display, status])
    btn_segment.click(segment_slice, outputs=[result_display, status])
    btn_propagate.click(propagate_volume, outputs=[result_display, status])
    slice_slider.change(show_propagated_slice, inputs=slice_slider, outputs=result_display)

# share=True evita imágenes "rotas": el iframe embebido del notebook no siempre
# puede llegar al servidor local (127.0.0.1), así que se usa un túnel público de Gradio.
demo.launch(inline=True, share=True, debug=False, height=900)


## Comparación cuantitativa contra el *ground truth*

Task09_Spleen solo incluye máscara de referencia para el **bazo**, así que la comparación cuantitativa de abajo evalúa específicamente el **Objeto 1** (el color rojo en la interfaz). Si segmentaste otros órganos con los objetos 2-4, esas máscaras son visualmente correctas pero no tienen *ground truth* en este dataset para calcular Dice.

::: {.callout-warning}
## Advertencia

Esta celda asume que ya propagaste el Objeto 1 al menos una vez desde la interfaz anterior. Si no hay máscaras para ese objeto, volvé a la celda de arriba, agregá un prompt sobre el bazo con el Objeto 1 activo y presioná **Propagar por el volumen**.
:::

In [ ]:
def dice_score(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    denom = pred.sum() + gt.sum()
    return 2 * np.logical_and(pred, gt).sum() / denom if denom > 0 else 1.0


spleen_masks = propagated_masks.get(1, {})
if not spleen_masks:
    print("Todavía no propagaste el Objeto 1 (bazo): usá la interfaz de arriba primero.")
else:
    gt_slices = lbl_vol[:, :, slice_start : slice_end + 1] > 0
    gt_slices = np.rot90(gt_slices, axes=(0, 1))  # misma rotación aplicada a los frames exportados

    per_slice_dice = [dice_score(mask, gt_slices[:, :, idx]) for idx, mask in spleen_masks.items()]
    pred_volume = np.stack([spleen_masks[i] for i in sorted(spleen_masks)], axis=-1)
    gt_volume = gt_slices[:, :, sorted(spleen_masks)]

    print(f"Dice promedio por corte (zero-shot, MedSAM2, Objeto 1 = bazo): {np.mean(per_slice_dice):.3f}")
    print(f"Dice volumétrico (zero-shot, MedSAM2, Objeto 1 = bazo):        {dice_score(pred_volume, gt_volume):.3f}")


En el `Notebook1` entrenamos una 3D U-Net **supervisada** sobre decenas de casos de este mismo dataset para segmentar el bazo, y en el `Notebook6` vimos cómo nnU-Net auto-configura ese pipeline completo. Acá, en cambio, MedSAM2 nunca vio máscaras de bazo durante el entrenamiento y solo recibió uno o dos prompts en un único corte.

El punto no es determinar cuál Dice es mayor, sino notar que se trata de **dos workflows distintos** (ver [clase4.qmd](../clase4.qmd)): uno que automatiza por completo la delineación de una estructura conocida, y otro que depende de una interacción puntual del experto pero no requiere haber entrenado un modelo específico para esa estructura.

## Extensión opcional: el mismo mecanismo sobre un video real

::: {.callout-note}
## Opcional

Esta sección es opcional y puede omitirse por tiempo. Su único objetivo es mostrar que `video_predictor` no distingue entre "cortes de TC" y "frames de un video real": el mecanismo de prompt + propagación es idéntico.
:::

Para probarlo con un video propio (por ejemplo, un clip corto de laparoscopía o ecografía):

1. Extraé los frames del video como JPEGs numerados (`ffmpeg -i video.mp4 frames/%05d.jpg`).
2. Apuntá `frames_dir` a esa carpeta.
3. Volvé a correr las celdas de "Construcción de los predictores" y la interfaz interactiva: no hace falta cambiar ni una línea de código, porque MedSAM2 ya trata ambos casos de la misma manera.